# Génération de format de chargement de tarifs

Génération de deux fichiers parquet pour les tarifs et pour les liens tarif-pdc.

In [ ]:
from datetime import datetime, timezone
import sys
from pathlib import Path
import json
import pandas as pd
# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))
from source.tariff import TariffObject, TariffDimensionTypeEnum, TariffElement, TariffElements, PriceComponent

In [ ]:
with open("../source/schema.json") as f:
    schema = json.load(f)

statiques = pd.read_csv("../data/qualicharge/donnees_statiques___liste_stations_pdcs_puissance_2026-06-30T22_54.csv", sep=",")
statiques_50 = statiques[statiques["puissance_nominale"] >= 50]
total_50 = len(statiques_50)
total_50

In [ ]:
def tariffs_to_dataframe(tariffs: list[dict], schema: dict, dict_tarif: bool = False, verbose: bool = False, correction: dict = None) -> pd.DataFrame:
    original_id = []
    original_last_updated = []
    raw = []
    start= []
    end = []
    if isinstance(tariffs, dict):
        tariffs = [tariffs]
    for tariff in tariffs:
        if correction:
            if "country_code" not in tariff:
                tariff["country_code"] = correction.get("country_code", "FR")
            if "party_id" not in tariff:
                tariff["party_id"] = correction.get("party_id", "XXX")
            if "type" in tariff:
                tariff["type"] = correction.get("type", "AD_HOC_PAYMENT")
        tarif = tariff['tariff'] if dict_tarif else tariff
        assert TariffObject.is_valid_json(schema, tarif, verbose=verbose), f"Tariff {tarif['id']} is not valid according to the schema"
        original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
        original_last_updated.append(tarif["last_updated"])
        raw.append(tarif)
        start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime(1970, 1, 1, tzinfo=timezone.utc).isoformat()))))
        end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
    return pd.DataFrame({
        "original_id": original_id,
        "original_last_updated": original_last_updated,
        "raw": raw,
        "start": start,
        "end": end})

In [ ]:
def tariffs_from_OCPI(locations: list, tariffs: list, schema: dict, correction: dict = None, verbose: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    tariff = tariffs_to_dataframe(tariffs, schema, correction=correction)  
    id_pdc_itinerance = []
    id_tariff = []
    for location in locations:
        if verbose:
            print(f"Processing location {location['id']}")
        if correction:
            if "country_code" not in location:
                location["country_code"] = correction.get("country_code", "FR")
            if "party_id" not in location:
                location["party_id"] = correction.get("party_id", "XXX")        
        for evse in location["evses"]:
            id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
            tariff_id = location["country_code"] + location["party_id"]
            if "tariff_ids" in evse["connectors"][0]:
                tariff_id += evse["connectors"][0]["tariff_ids"][0]
            else:
                tariff_id += evse["connectors"][0]["tariff_id"]
            if tariff_id not in tariff["original_id"].values:
                print(f"Tariff ID {tariff_id} not found in original tariffs")
            id_tariff.append(tariff_id)
    tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": id_pdc_itinerance,
        "id_tariff": id_tariff})
    return (tariff, tariffpdc)

In [ ]:
def tariff_simple(country_code: str, party_id: str, id: str, last_updated: str, dimensions: list[str], prices: list[float], start_date_time: str=None, end_date_time: str=None):
    price_components = []
    for i in range(len(dimensions)):
        price_component = PriceComponent(type=TariffDimensionTypeEnum(dimensions[i]), price=prices[i], step_size=1)
        price_components.append(price_component)
    tariff_elements = TariffElements(
        root=[TariffElement(price_components=price_components)]
    )
    tariff = TariffObject(
        country_code=country_code,
        party_id=party_id,
        id=id,
        elements=tariff_elements,
        last_updated=last_updated,
        start_date_time=start_date_time,
        end_date_time=end_date_time
    )
    return json.loads(tariff.model_dump_json(exclude_none=True))

In [ ]:
def tariffs_to_text(tariffs: pd.DataFrame, operator: str) -> None:
    text = "# Tarifs au format texte des tarifs " + operator + "\n\n"
    for tarif in tariffs["raw"]:
        tariff = TariffObject.model_validate(tarif)
        text += tariff.to_text() + "\n"
    with open(f"../data/{operator}/tariffs_text.md", "w", encoding="utf-8") as f:
        f.write(text)

In [ ]:
def export_tariffs(tariffs: pd.DataFrame, tariffpdc: pd.DataFrame, operator: str, display: bool = False) -> None:
    tariffpdc = tariffpdc.dropna().drop_duplicates(subset=["id_pdc_itinerance", "id_tariff"], keep="last")
    tariffpdc_50 = tariffpdc[tariffpdc["id_pdc_itinerance"].isin(statiques_50["id_pdc_itinerance"])]
    print(f"len tariff, tariffpdc, tariffpdc_50 et % : {len(tariffs)}, {len(tariffpdc)}, {len(tariffpdc_50)}, {len(tariffpdc_50)/total_50*100:.2f} %")
    print(f"len unique tariffpdc, tariffpdc_50 : {tariffpdc['id_pdc_itinerance'].nunique()}, {tariffpdc_50['id_pdc_itinerance'].nunique()}")
    tariffs_to_text(tariffs, operator)
    tariffs.to_parquet(f"../data/{operator}/qualicharge_tariff.parquet", index=False)
    tariffpdc.to_parquet(f"../data/{operator}/qualicharge_tariffpdc.parquet", index=False)
    tariffpdc.to_csv(f"../data/{operator}/qualicharge_tariffpdc.csv", index=False)
    if display:
        print(tariffs["raw"][0], "\n")
        print(tariffs, "\n")
        print(tariffpdc)

## Monta

In [ ]:
import ast

operator = "monta"


tariffs = pd.read_csv(f"../data/{operator}/NAP FR - Tariffs.csv", sep=",")
tariffs.columns = tariffs.columns.str.lower()
tariffs["elements"] = tariffs["elements"].str.replace('"restrictions": null,', '').str.replace('priceComponents', 'price_components').str.replace('stepSize', 'step_size').apply(ast.literal_eval)
tariffs["last_updated"] = pd.to_datetime(tariffs["last_updated"], utc=True).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
tariffs.rename(columns={"tariff_identifier": "id", "tariff_type": "type"}, inplace=True)
tariffs = tariffs.to_dict(orient="records")
tariffs = [{k: v for k, v in d.items() if isinstance(v, list) or pd.notna(v)} for d in tariffs]
tariffs = tariffs_to_dataframe(tariffs, schema, verbose=False)

tariffpdc = pd.read_csv(f"../data/{operator}/NAP FR - Tariff Assignments.csv", sep=",") # .rename(columns={"rateid": "id_tariff"})
tariffpdc.rename(columns={"TARIFF_IDENTIFIER": "id_tariff", "ID_PDC_ITINERANCE": "id_pdc_itinerance"}, inplace=True)

export_tariffs(tariffs, tariffpdc, operator)

## Tesla

In [ ]:
operator = "tesla"

with open(f'../data/{operator}/FR_PTSCH_locations_06-23.json', 'r', encoding='utf-8') as f:
    tesla_locations = json.load(f)
with open(f'../data/{operator}/FR_PTSCH_Tariffs_06-23.json', 'r', encoding='utf-8') as f:
    tesla_tariffs = json.load(f)
tariffs, tariffpdc = tariffs_from_OCPI(tesla_locations["data"], tesla_tariffs["data"], schema, correction = {"country_code": "NL", "party_id": "TSL"})

export_tariffs(tariffs, tariffpdc, operator)

## Fastned

In [ ]:
operator = "fastned"
country_code = "FR"
party_id = "FAS"
id = "tarif_unique_1"

tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2024-06-01T12:00:00Z",
    dimensions=["ENERGY"],
    prices=[0.5083]) # 0.61€/kWh TTC -> 0.61/1.2 = 0.5083€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_fastned = statiques[statiques["id_pdc_itinerance"].str.startswith("FRFAS", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_fastned["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_fastned)})

export_tariffs(tariffs, tariffpdc, operator)

## Atlante

In [ ]:
operator = "atlante"
country_code = "FR"
party_id = "ATL"

tariffs_pdc = pd.read_csv(f"../data/{operator}/current_tariffs.csv", sep=",") # .rename(columns={"rateid": "id_tariff"})
tariffs_pdc["id_tariff"] = tariffs_pdc["rateid"].apply(lambda x: "FRATL" + x)
tariffpdc = tariffs_pdc[["id_pdc_itinerance", "id_tariff"]]

tariffs = tariffs_pdc.drop_duplicates(subset=["id_tariff"]).reset_index(drop=True)
tariffs["original_id"] = tariffs["id_tariff"]
tariffs["original_last_updated"] = tariffs["rate_date_modified"]
tariffs["raw"] = tariffs.apply(lambda row: tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=row["rateid"],
    last_updated=row["rate_date_modified"] + "Z",
    dimensions=["ENERGY", "TIME", "PARKING_TIME"],
    prices=[row["costkwh"]/1.2, row["costchargingtime"]/1.2, row["costparkingtime"]/1.2]), axis=1)
tariffs["start"] = tariffs["rate_date_modified"]
tariffs["end"] = None
tariffs = tariffs[["original_id", "original_last_updated", "raw", "start", "end"]]

export_tariffs(tariffs, tariffpdc, operator)

## NW IEcharge

In [ ]:
operator = "iecharge"
country_code = "FR"
party_id = "IEN"
id = "TARIFF_IECHARGE"

tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2026-07-01T14:41:00Z",
    dimensions=["ENERGY"],
    prices=[0.2083]) # 0.25€/kWh TTC -> 0.25/1.2 = 0.2083€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_iecharge = statiques[statiques["id_pdc_itinerance"].str.startswith("FRIEN", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_iecharge["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_iecharge)})

export_tariffs(tariffs, tariffpdc, operator)

## Lidl

In [ ]:
operator = "lidl"
country_code = "FR"
party_id = "LDL"
id = "TARIFF_LIDL_PLUS_DC"

tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2026-04-22T14:41:00Z",
    dimensions=["ENERGY"],
    prices=[0.325]) # 0.39€/kWh TTC -> 0.39/1.2 = 0.325€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_lidl = statiques[statiques["id_pdc_itinerance"].str.startswith("FRLDL", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_lidl["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_lidl)})

export_tariffs(tariffs, tariffpdc, operator)

## R3

In [ ]:
operator = "R3"
country_code = "FR"
party_id = "R3M"
id = "TARIFF_R3_DC"

tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2026-06-24T14:41:00Z",
    dimensions=["ENERGY"],
    prices=[0.46]) # 0.46€HT/kWh
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_R3 = statiques[statiques["id_pdc_itinerance"].str.startswith("FRR3M", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_R3["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_R3)})

export_tariffs(tariffs, tariffpdc, operator)

## Total

In [ ]:
operator = "total"

with open(f"../data/{operator}/tariffs_qualicharge 1.json", 'r', encoding='utf-8') as f:
    total_tarifs = json.load(f)
tariffs = tariffs_to_dataframe(total_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]

tariffs_pdc = pd.read_csv(f"../data/{operator}/lien_tariff.csv", sep=",")
tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="id_tariff", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})

export_tariffs(tariffs, tariffpdc, operator)

## Izivia

In [ ]:
operator = "izivia"

izivia_tarifs = []
jsons = Path(f"../data/{operator}/jsons")
for file in jsons.iterdir():
    with open(file, 'r', encoding='utf-8') as f:
        izivia_tarifs.append(json.load(f))

tariffs = tariffs_to_dataframe(izivia_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]

tariffs_pdc = pd.read_csv(f"../data/{operator}/edf-bs-tariffs-by-evse.csv", sep=";")
tariffs_pdc.rename(columns={"id_point_de_charge": "id_pdc_itinerance", "tariff_id": "id_tariff"}, inplace=True)

tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="id_tariff", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})

export_tariffs(tariffs, tariffpdc, operator)


## BP

In [ ]:
operator = "BP"

with open(f'../data/{operator}/bp_tariff.json', 'r', encoding='utf-8') as f:
    bp_tariff = json.load(f)

tariffs = tariffs_to_dataframe([bp_tariff], schema, verbose=True)
statiques_BP = statiques[statiques["id_pdc_itinerance"].str.startswith("FRBPE", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_BP["id_pdc_itinerance"].tolist(),
        "id_tariff": [tariffs["original_id"][0]] * len(statiques_BP)})

export_tariffs(tariffs, tariffpdc, operator)

## Driveco

In [ ]:
operator = "driveco"
country_code = "FR"
party_id = "DRV"

with open(f'../data/{operator}/data-locations-driveco.json', 'r', encoding='utf-8') as f:
    driveco_locations = json.load(f)
with open(f'../data/{operator}/data-tariffs-driveco.json', 'r', encoding='utf-8') as f:
    driveco_tariffs = json.load(f)
locations_json = driveco_locations["data"]
for location in locations_json:
    location["country_code"] = country_code if "country_code" not in location else location["country_code"]
    location["party_id"] = party_id if "party_id" not in location else location["party_id"]
    # correction : tariff_id : value -> tariff_ids : [value]
    for evse in location["evses"]:
        for connector in evse["connectors"]:
            if "tariff_ids" not in connector:
                connector["tariff_ids"] = [connector["tariff_id"]]
tariffs_json = driveco_tariffs["data"]
for tarif in tariffs_json:
    tarif["country_code"] = country_code if "country_code" not in tarif else tarif["country_code"]
    tarif["party_id"] = party_id if "party_id" not in tarif else tarif["party_id"]
    # correction : step_size= 0 -> step_size= 1
    for element in tarif["elements"]:
        for price_component in element["price_components"]:
            if price_component["step_size"] == 0:
                price_component["step_size"] = 1
tariffs, tariffpdc = tariffs_from_OCPI(driveco_locations["data"], tariffs_json, schema)

export_tariffs(tariffs, tariffpdc, operator)

## Easycharge

In [ ]:

operator = "easycharge"

with open(f'../data/{operator}/EASYCHARGE_tarifs_FRTDA-FREBN-FRECH_30062026.json', 'r', encoding='utf-8') as f:
    easycharge_tarifs = json.load(f)
tariffs = tariffs_to_dataframe(easycharge_tarifs, schema, dict_tarif=True, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

easycharge_pdcs = pd.read_csv(f"../data/{operator}/EASYCHARGE_tarifs vs pdc.csv", sep=",")
tariffs_pdc = easycharge_pdcs[["id_pdc_itinerance", "tariff_id"]].rename(columns={"tariff_id": "id_tariff"})
print("len(tariffs_pdc):", len(tariffs_pdc), "\n")

tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="id_tariff", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
#print(tariff_pdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)

## Ionity

In [ ]:

operator = "ionity"

with open(f"../data/{operator}/IONITY_tariffs_France_v2.json", 'r', encoding='utf-8') as f:
    ionity_tarifs = json.load(f)
tariffs = tariffs_to_dataframe(ionity_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

tariffs_pdc = pd.read_csv(f"../data/{operator}/Tariffs CSV - IONITY France - Connectors download.csv", sep=",")
tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="src_id", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
#print(tariff_pdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)

## Electra

In [ ]:

operator = "electra"

with open(f"../data/{operator}/Electra_tariffs_France_v2.json", 'r', encoding='utf-8') as f:
    electra_tarifs = json.load(f)
tariffs = tariffs_to_dataframe(electra_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

tariffs_pdc = pd.read_csv(f"../data/{operator}/Tariffs CSV - Electra France - Connectors download v2.csv", sep=",")
if "id_pdc_itinerance" not in tariffs_pdc.columns:
    tariffs_pdc["id_pdc_itinerance"] = tariffs_pdc["evse_id"].str.replace(r"[ *\s]", "", regex=True)
#print(tariffs_pdc.sort_values(by='id_pdc_itinerance'), "\n")
tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="src_id", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
#print(tariff_pdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)

## Shell

In [ ]:
operator = "shell"

with open(f"../data/{operator}/Shell_Recharge_tariffs_France.json", 'r', encoding='utf-8') as f:
    shell_tariffs = json.load(f)
tariffs = tariffs_to_dataframe(shell_tariffs, schema, verbose=False, correction = {"type": "AD_HOC_PAYMENT"})
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

tariffs_pdc = pd.read_csv(f"../data/{operator}/Tariffs CSV - Shell Recharge France - Connectors download.csv", sep=",")
if "id_pdc_itinerance" not in tariffs_pdc.columns:
    tariffs_pdc["id_pdc_itinerance"] = tariffs_pdc["evse_id"].str.replace(r"[ *\s]", "", regex=True)
tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="src_id", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
#print(tariff_pdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)